# Index

In [54]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
from index.inverted_index import Index
from index.documents import load_dataset, tokenizer


# index = Index().build_index(rcv1, tokenizer)

In [56]:
# rcv1 = load_dataset("rcv1")

In [57]:
# i = 0
# x = rcv1.data[i]
# y = rcv1.target[i]
# sid = rcv1.sample_id[i]

# print("sid", sid)
# print("x", x)
# print("y", y)

In [58]:
# from sklearn.cluster import MiniBatchKMeans

# X = rcv1.data # CSR matrix (804414 x 47236)
# mbk = MiniBatchKMeans(n_clusters=20, batch_size=1000, random_state=0)
# mbk.fit(X)                          # supports sparse input

# # cluster centers are dense (20 x n_features)
# centers = mbk.cluster_centers_
# # top features per cluster
# import numpy as np
# for i, c in enumerate(centers):
#     top_idxs = np.argsort(-c)[:10]
#     print("cluster", i, "top feature indices:", top_idxs)

In [59]:
msmarco = load_dataset("msmarco-passage/dev/judged")

In [60]:
print(msmarco)

Dataset(id='msmarco-passage/dev/judged', provides=['docs', 'queries', 'qrels'])


In [61]:
import locale, sys
print(sys.flags.utf8_mode)                 # should be 1
print(locale.getpreferredencoding(False))  # should be UTF-8

1
UTF-8


In [62]:
import numpy as np
import random

class OrderedDataset:
    def __init__(self, docs, order):
        self.docs = docs
        self.order = order
    def docs_iter(self):
        for i in self.order:
            yield self.docs[i-1]


NUM_DOCS = 10000

docs = []
for i, doc in enumerate(msmarco.docs_iter()):
    docs.append(doc)
    if i == NUM_DOCS - 1:
        break

order = list(range(1, NUM_DOCS + 1))
random.shuffle(order)

random_dataset = OrderedDataset(docs, order)

# for doc in random_dataset.docs_iter():
#     print(doc)


unordered_index = Index()
unordered_index.build_index(random_dataset, tokenizer, max_num_docs=NUM_DOCS)



{'postacute': [(1, 2), (322, 1), (1460, 1), (3857, 1), (6710, 1)],
 'withdrawal': [(1, 3),
  (322, 1),
  (345, 1),
  (869, 1),
  (1460, 2),
  (2739, 4),
  (2829, 1),
  (3713, 3),
  (3857, 1),
  (5348, 1),
  (6710, 4),
  (6776, 1),
  (9651, 3),
  (9685, 3),
  (9749, 3)],
 'symptoms': [(1, 2),
  (12, 1),
  (56, 1),
  (265, 1),
  (278, 1),
  (303, 1),
  (316, 2),
  (322, 1),
  (422, 1),
  (493, 1),
  (739, 2),
  (837, 3),
  (878, 1),
  (914, 2),
  (1019, 2),
  (1152, 1),
  (1209, 3),
  (1266, 3),
  (1276, 2),
  (1352, 1),
  (1445, 1),
  (1460, 3),
  (1467, 1),
  (1477, 2),
  (1515, 1),
  (1565, 2),
  (1844, 2),
  (2016, 1),
  (2136, 1),
  (2187, 3),
  (2227, 1),
  (2290, 3),
  (2319, 1),
  (2739, 2),
  (2757, 1),
  (2765, 1),
  (2939, 1),
  (2996, 1),
  (3008, 1),
  (3017, 1),
  (3068, 2),
  (3071, 1),
  (3239, 1),
  (3309, 2),
  (3348, 1),
  (3357, 1),
  (3507, 1),
  (3519, 1),
  (3619, 1),
  (3652, 1),
  (3857, 1),
  (3915, 2),
  (3948, 2),
  (3987, 1),
  (4073, 1),
  (4463, 1),
  (4514

In [63]:
from scipy.sparse import csr_matrix
import numpy as np

def calculate_tf_idf(index, doc_id):
    tf_idf_vector = {}
    for term in index.get_terms():
        tf = index.get_term_frequency(term, doc_id)
        idf = index.get_inverse_document_frequency(term)
        tf_idf_vector[term] = tf * idf
    return tf_idf_vector

def create_tf_idf_matrix(index):
    tf_idf_matrix = {}
    for doc_id in index.get_doc_ids():
        tf_idf_matrix[doc_id] = calculate_tf_idf(index, doc_id)
    return tf_idf_matrix

def create_sparse_tf_idf_matrix(index, dtype=np.float32):
    terms = list(index.get_terms())
    term_to_col = {t:i for i,t in enumerate(terms)}
    doc_ids = index.get_doc_ids()
    doc_to_row = {did:i for i,did in enumerate(doc_ids)}

    rows, cols, data = [], [], []
    for term, posting in index.index.items():
        idf = index.get_inverse_document_frequency(term)
        col = term_to_col[term]
        for doc_id, tf in posting:
            rows.append(doc_to_row[doc_id])
            cols.append(col)
            data.append(np.log10(1+tf) * idf)

    data = np.array(data, dtype=dtype)
    mat = csr_matrix((data, (rows, cols)), shape=(len(doc_ids), len(terms)), dtype=dtype)
    return mat, doc_ids, terms

In [64]:
# tf_idf_matrix = create_tf_idf_matrix(msmarco_index)
# print("TF-IDF vector for document 1:", tf_idf_matrix[1])
# print("tf idf values", list(tf_idf_matrix.values().__iter__())[:10])

mat, doc_ids, terms = create_sparse_tf_idf_matrix(unordered_index)
print("Sparse TF-IDF matrix shape:", mat.shape)
print("Document IDs:", doc_ids[:10])
print("Terms:", terms[:10])

# bytearray(b'\x00\x01\x02')

Sparse TF-IDF matrix shape: (10000, 37675)
Document IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Terms: ['postacute', 'withdrawal', 'symptoms', 'some', 'people', 'may', 'experience', 'a', 'prolonged', 'or']


In [65]:
import pandas as pd

# df = pd.DataFrame(tf_idf_matrix).T
# print("TF-IDF DataFrame shape:", df.shape)

df = pd.DataFrame.sparse.from_spmatrix(mat, index=doc_ids, columns=terms)
print("DF shape:", df.shape)

DF shape: (10000, 37675)


In [66]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

norm_df = normalize(df, norm="l2") #! cosa fa , è fondamentele se non la metti clusteing fa schifo 

kmeans = KMeans(n_clusters=50, random_state=0) #! use jaccard distance instead of euclidean

clusters = kmeans.fit(norm_df)
print("Cluster centers:", kmeans.cluster_centers_)

document_clusters = kmeans.labels_
print("Document clusters:", document_clusters)

Cluster centers: [[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.00280309 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
Document clusters: [32 15  9 ...  6 14 42]


In [67]:
medoids = []

for i in range(kmeans.n_clusters):
    cluster_docs = np.flatnonzero(document_clusters == i)
    cluster_vectors = norm_df[cluster_docs]
    medoid_idx = np.argmin(np.linalg.norm(cluster_vectors - kmeans.cluster_centers_[i], axis=1))
    medoids.append(cluster_docs[medoid_idx])

# medoid_vectors = norm_df[medoids].toarray()

medoid_vectors = norm_df[medoids]
nonzero_cols = medoid_vectors.getnnz(axis=0) > 0
medoid_vectors = medoid_vectors[:, nonzero_cols].toarray()

print("Medoids:", medoids)
print("Medoid vectors shape:", medoid_vectors.shape)
print("Medoid vectors:", medoid_vectors)

# for i in range(10):
#     cluster_docs = [doc_id for doc_id, cluster in enumerate(document_clusters) if cluster == i]
#     new_docs_ordering[i] = cluster_docs
#     print(f"Cluster {i} has {len(cluster_docs)} documents.")

# from itertools import islice

# # medoids contiene row index della matrice (0-based)
# target_rows = set(medoids)
# medoid_docs = {}

# for row_idx, doc in enumerate(islice(msmarco.docs_iter(), len(doc_ids))):
#     if row_idx in target_rows:
#         medoid_docs[row_idx] = doc
#     if len(medoid_docs) == len(target_rows):
#         break

# # esempio: testo del primo medoid
# print(medoid_docs[medoids[0]].text)

# # opzionale: stampa tutti i medoid
# for m in medoids:
#     print(f"\nMedoid row {m}")
#     print(medoid_docs[m].text[:500])

Medoids: [np.int64(7992), np.int64(4069), np.int64(7915), np.int64(2788), np.int64(6075), np.int64(101), np.int64(5216), np.int64(3935), np.int64(2249), np.int64(3311), np.int64(8009), np.int64(5380), np.int64(1587), np.int64(1102), np.int64(3848), np.int64(5056), np.int64(4887), np.int64(1779), np.int64(3183), np.int64(5840), np.int64(8952), np.int64(5658), np.int64(2232), np.int64(6989), np.int64(7251), np.int64(6556), np.int64(8275), np.int64(5901), np.int64(6517), np.int64(8919), np.int64(1494), np.int64(8680), np.int64(3504), np.int64(144), np.int64(8994), np.int64(8565), np.int64(5752), np.int64(9344), np.int64(315), np.int64(9445), np.int64(602), np.int64(6349), np.int64(9538), np.int64(9932), np.int64(8310), np.int64(8726), np.int64(5636), np.int64(535), np.int64(3223), np.int64(7387)]
Medoid vectors shape: (50, 1139)
Medoid vectors: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0

In [68]:
from sklearn.metrics import pairwise_distances
import numpy as np

def calculate_distance_matrix(norm_df, medoid_vectors, method="euclidean"):
    # medoid_vectors = norm_df[medoids]  # CSR (k, d)

    if method == "euclidean":
        return pairwise_distances(medoid_vectors, metric="euclidean")
    elif method == "cosine":
        return pairwise_distances(medoid_vectors, metric="cosine")
    elif method == "jaccard":
        # Jaccard in sklearn works on boolean/binary features
        return pairwise_distances((medoid_vectors > 0).astype(np.uint8), metric="jaccard")
    else:
        raise ValueError(f"Unknown method: {method}")

In [69]:
# medoid_vectors = norm_df[medoids]
# print("Medoid vectors shape:", medoid_vectors.shape)
# print("Medoid vectors:", medoid_vectors)
# medoid_vectors.toarray()  
# print("Medoid vectors:", medoid_vectors)

dist_matrix = calculate_distance_matrix(norm_df, medoid_vectors, method="jaccard")
print("Distance matrix shape:", dist_matrix.shape)
print("Distance matrix:\n", dist_matrix)

Distance matrix shape: (50, 50)
Distance matrix:
 [[0.         0.93670886 0.93055556 ... 0.97183099 0.95348837 0.925     ]
 [0.93670886 0.         0.90540541 ... 0.9        0.96703297 0.88888889]
 [0.93055556 0.90540541 0.         ... 0.87096774 0.93902439 0.92207792]
 ...
 [0.97183099 0.9        0.87096774 ... 0.         0.97530864 0.93243243]
 [0.95348837 0.96703297 0.93902439 ... 0.97530864 0.         0.93333333]
 [0.925      0.88888889 0.92207792 ... 0.93243243 0.93333333 0.        ]]


c:\Users\Utente\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2462: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [70]:
import networkx as nx
tsp = nx.approximation.traveling_salesman_problem

tsp_graph = nx.from_numpy_array(dist_matrix)
tsp_path = tsp(tsp_graph, cycle=False)
print("TSP path (medoid indices):", tsp_path)


TSP path (medoid indices): [34, 35, 17, 20, 16, 42, 30, 44, 48, 36, 29, 49, 45, 13, 9, 19, 37, 12, 43, 11, 40, 2, 24, 3, 5, 23, 47, 22, 32, 6, 41, 1, 33, 25, 15, 10, 21, 18, 0, 28, 27, 14, 31, 46, 38, 4, 8, 39, 26, 7]


In [71]:
def reorder_documents_by_tsp(document_clusters, medoids, tsp_path):
    """
    Reorder documents by cluster order induced by TSP over medoids.
    Inside each cluster, keep documents in their original order (row order).

    Args:
        document_clusters: array-like of shape (n_docs,), cluster label per doc.
        medoids: list of medoid doc indices (not required for ordering here, kept for API compatibility).
        tsp_path: list of cluster ids in the order returned by TSP.

    Returns:
        ordered_doc_ids: list of document row indices in final order.
        cluster_to_docs: dict {cluster_id: [doc_ids in original order]}.
    """
    document_clusters = np.asarray(document_clusters)

    # Build cluster -> docs preserving natural/original order
    cluster_to_docs = {
        c: np.flatnonzero(document_clusters == c).tolist()
        for c in range(len(medoids))
    }

    # Concatenate clusters following TSP order
    ordered_doc_ids = []
    for c in tsp_path:
        ordered_doc_ids.extend(cluster_to_docs.get(c, []))

    return ordered_doc_ids, cluster_to_docs

In [72]:
import random
ordered_doc_ids, cluster_docs = reorder_documents_by_tsp(document_clusters, medoids, tsp_path)
# print("First 30 docs in reordered list:", ordered_doc_ids[:30])
# print(len(ordered_doc_ids), "documents reordered.")



In [73]:
# DEBUG, checking the documents in the clusters


sample_docs = []

for i, doc in enumerate(msmarco.docs_iter()):
  #if(i < 100):
    #print(doc)
  sample_docs.append(doc)
  # print(f"Document {i}: {doc}")
  if i == NUM_DOCS:
    break


first_cluster = tsp_path[1]

print(f"\nDocuments in first cluster (cluster {first_cluster}):")

for i, doc_id in enumerate(cluster_docs[first_cluster][:100]):
    # print(f"{doc_id} sample doc reference: {sample_docs[doc_id]}")
    if sample_docs[doc_id] is not None: # maybe useless
        doc = sample_docs[doc_id]
        print(f"Doc ID {doc.doc_id}: {doc.text}...")


Documents in first cluster (cluster 35):
Doc ID 57: More concentrated urine in the bladder can be darker. As long as your urine returned to a more-normal, light yellow color after you drank more water, there is no need to be concerned....
Doc ID 182: Importantly, there are also clear opportunities to use video games as a way to empower young people to manage their mental health and wellbeing, and potentially circumvent psychological distress. An important thing is that players (and their parents) engage thoughtfully with what they are playing.n a randomised controlled trial with a clinically depressed sample of adults, the positive influences of video games have been shown to include a reduction in tension, anger, depression and fatigue and increase in vigour....
Doc ID 208: There are three types of tundra: arctic tundra, alpine tundra, and Antarctic tundra. In tundra, the vegetation is composed of dwarf shrubs, sedges and grasses, mosses, and lichens.Scattered trees grow in some tund

In [74]:
from itertools import islice

docs = list(islice(msmarco.docs_iter(), len(document_clusters)))

class OrderedDataset:
    def __init__(self, docs, order):
        self.docs = docs
        self.order = order
    def docs_iter(self):
        for i in self.order:
            yield self.docs[i]

ordered_dataset = OrderedDataset(docs, ordered_doc_ids)
reordered_inverted_index = Index()
reordered_inverted_index.build_index(
    ordered_dataset,
    tokenizer,
    max_num_docs=-1, #! -1 since reordered docs are already limited to 10001 in the original index
    d_gap=True
)

stat_r = reordered_inverted_index.get_approx_size()

unordered_index = Index()
unordered_index.build_index(random_dataset, tokenizer, max_num_docs=NUM_DOCS)

# msmarco_index = Index()
# msmarco_index.build_index(unordered_index, tokenizer, max_num_docs=1000, d_gap=True)

stat_m = unordered_index.get_approx_size()
print(f"Original index size: {stat_m} bytes")
print(f"Reordered index size: {stat_r} bytes")

ind_1 = reordered_inverted_index.index
ind_2 = unordered_index.index

print("Msmarco index size:", len(unordered_index.index))


=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      12,172,320
Total bits:   24,344,640
Total bytes:  3,043,080.00

=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      12,172,320
Total bits:   24,344,640
Total bytes:  3,043,080.00
Original index size: {'term_bytes': 0, 'doc_id_bits': 12172320, 'tf_bits': 12172320, 'total_bits': 24344640, 'total_bytes': 3043080.0} bytes
Reordered index size: {'term_bytes': 0, 'doc_id_bits': 12172320, 'tf_bits': 12172320, 'total_bits': 24344640, 'total_bytes': 3043080.0} bytes
Msmarco index size: 37675


In [75]:
def all_gaps(index):
    return [
        gap
        for posting_list in index.index.values()
        for gap in posting_list
    ]

gaps_order = all_gaps(reordered_inverted_index)

gaps_unorderd = all_gaps(unordered_index)


print("AVG ordered:", np.mean(gaps_order))
print("AVG unordered:", np.mean(gaps_unorderd))

print("MEDIAN ordered:", np.median(gaps_order))
print("MEDIAN unordered:", np.median(gaps_unorderd))

AVG ordered: 323.040068614693
AVG unordered: 2498.2277744916337
MEDIAN ordered: 2.0
MEDIAN unordered: 10.0


In [49]:
# msmarco_index = Index()
# msmarco_index.clear_index()
# msmarco_index.build_index(msmarco, tokenizer, max_num_docs=10000, d_gap=True)
# print("Msmarco index size:", len(msmarco_index.index))
# msmarco_index.remove_tf()
# msmarco_index.compress_index("gamma")
# msmarco_index.save_index("msmarco_index.bin")

In [50]:
# test_index = Index()
# test_index.clear_index()
# test_index.load_index("msmarco_index.bin")

In [51]:
# print("Test index loaded. Number of terms:", len(test_index.index))

In [52]:
def check_correctness(original_index, loaded_index):
    i = 0
    for term, posting in original_index.index.items():
        # if term not in loaded_index.index:
        #     print(f"Term '{term}' missing in loaded index.")
        #     return False
        st = str(i)
        if posting != loaded_index.index[st]:
            print(f"Posting list mismatch for term '{term}'.")
            print(f"Original: {posting}")
            print(f"Loaded: {loaded_index.index[st]}")
            return False
        i += 1
    print("Original and loaded indices match.")
    return True

# test_index.compress_index("gamma")  # Decompress the loaded index for comparison
# check_correctness(msmarco_index, test_index)

In [26]:
# Test compressions
import copy

def prepare_index(src_index, method):
    idx = copy.deepcopy(src_index)
    idx.remove_tf()
    idx.compress_index(method)
    return idx

# Original index variants
gamma_index = prepare_index(unordered_index, "gamma")
delta_index = prepare_index(unordered_index, "delta")
vb_index = prepare_index(unordered_index, "varbyte")

# Reordered index variants
gamma_index_r = prepare_index(reordered_inverted_index, "gamma")
delta_index_r = prepare_index(reordered_inverted_index, "delta")
vb_index_r = prepare_index(reordered_inverted_index, "varbyte")

print("\n" + "=" * 70)
print(f"{'ORIGINAL (NOT REORDERED) INDEX':^70}")
print("=" * 70)
print("Uncompressed:")
unordered_index.remove_tf()  # Ensure TF is removed for size calculation
unordered_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index.get_approx_size()

print("\nDelta Compressed:")
delta_index.get_approx_size()

print("\nVarbyte Compressed:")
vb_index.get_approx_size()

print("\n" + "=" * 70)
print(f"{'REORDERED INDEX (TSP)':^70}")
print("=" * 70)
print("Uncompressed:")
reordered_inverted_index.remove_tf()  # Ensure TF is removed for size calculation
reordered_inverted_index.get_approx_size()

print("\nGamma Compressed:")
gamma_index_r.get_approx_size()

print("\nDelta Compressed:")
delta_index_r.get_approx_size()

print("\nVarbyte Compressed:")
vb_index_r.get_approx_size()


                    ORIGINAL (NOT REORDERED) INDEX                    
Uncompressed:

=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      0
Total bits:   12,172,320
Total bytes:  1,521,540.00

Gamma Compressed:

=== Index Size Breakdown ===
Doc ID bits:  9,024,455
TF bits:      0
Total bits:   9,024,455
Total bytes:  1,128,056.88

Delta Compressed:

=== Index Size Breakdown ===
Doc ID bits:  6,974,406
TF bits:      0
Total bits:   6,974,406
Total bytes:  871,800.75

Varbyte Compressed:

=== Index Size Breakdown ===
Doc ID bits:  6,047,232
TF bits:      0
Total bits:   6,047,232
Total bytes:  755,904.00

                        REORDERED INDEX (TSP)                         
Uncompressed:

=== Index Size Breakdown ===
Doc ID bits:  12,172,320
TF bits:      0
Total bits:   12,172,320
Total bytes:  1,521,540.00

Gamma Compressed:

=== Index Size Breakdown ===
Doc ID bits:  4,326,549
TF bits:      0
Total bits:   4,326,549
Total bytes:  540,818.62

Delta Compressed:

=== In

{'term_bytes': 0,
 'doc_id_bits': 4216744,
 'tf_bits': 0,
 'total_bits': 4216744,
 'total_bytes': 527093.0}

AVG ordered: 639.8453934829187
AVG unordered: 4996.774262917833
MEDIAN ordered: 50.0
MEDIAN unordered: 5000.0


In [23]:
from index.inverted_index import Index

test = Index()
num = 4
# res = test._Index__varbyte_encode(num)
# print(f"Varbyte encoding of {num}: {res}")
# for value in test._Index__varbyte_decode(res):
#     print(value)

# byte_val = int(res, 2)
# print(f"Decoded value: {byte_val}")

# res_2 = test._Index__gamma_encode(num)
# print(f"Gamma encoding of {num}: {res_2}")
# test._Index__gamma_decode(res_2)

# res_3 = test._Index__delta_encode(num)
# print(f"Delta encoding of {num}: {res_3}")
# test._Index__delta_decode(res_3)

# d_gaps_list = [3,4,7,21,9,12,2,17]
# res_4_list, outliers, min_ = test.pfor_delta_encode(d_gaps_list) 
# print(f"PForDelta encoding of {d_gaps_list}: {res_4_list}")
# print(f"Outliers: {outliers}")
# print(f"Minimum value: {min_}")
# #! how do i distinguish outliers from non-outliers in the decode? assuming that its encoded as contiguous block of bits

# print(test.pfor_delta_decode(res_4_list, outliers, min_))

In [20]:
#! test encoding and decoding

from index._compression import IndexCompressor

test = IndexCompressor()

numbers = [1, 2, 3, 4, 5,343, 1000, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
for num in numbers:
    res = test.varbyte_encode(num)
    print(f"Varbyte encoding of {num}: {res}")
    for value in test.varbyte_decode(res):
        print(value)

    res_2 = test.gamma_encode(num)
    print(f"Gamma encoding of {num}: {res_2}")
    for value in test.gamma_decode(res_2):
         print(value)
    # test.gamma_decode(res_2)

    res_3 = test.delta_encode(num)
    print(f"Delta encoding of {num}: {res_3}")
    for value in test.delta_decode(res_3):
        print(value)


Varbyte encoding of 1: bitarray('10000001')
1
Gamma encoding of 1: bitarray('0')
1
Delta encoding of 1: bitarray('0')
1
Varbyte encoding of 2: bitarray('10000010')
2
Gamma encoding of 2: bitarray('100')
2
Delta encoding of 2: bitarray('1000')
2
Varbyte encoding of 3: bitarray('10000011')
3
Gamma encoding of 3: bitarray('101')
3
Delta encoding of 3: bitarray('1001')
3
Varbyte encoding of 4: bitarray('10000100')
4
Gamma encoding of 4: bitarray('11000')
4
Delta encoding of 4: bitarray('10100')
4
Varbyte encoding of 5: bitarray('10000101')
5
Gamma encoding of 5: bitarray('11001')
5
Delta encoding of 5: bitarray('10101')
5
Varbyte encoding of 343: bitarray('0000001011010111')
343
Gamma encoding of 343: bitarray('11111111001010111')
343
Delta encoding of 343: bitarray('111000101010111')
343
Varbyte encoding of 1000: bitarray('0000011111101000')
1000
Gamma encoding of 1000: bitarray('1111111110111101000')
1000
Delta encoding of 1000: bitarray('1110010111101000')
1000
Varbyte encoding of 1024:

In [45]:
def check_correctness(original_index, loaded_index, with_tf=False):
    original_postings = [
        [gap for gap, _tf in posting_list] if not with_tf else posting_list
        for _, posting_list in original_index.index.items()
    ]
    loaded_postings = [
        posting_list
        for _, posting_list in loaded_index.index.items()
    ]

    if len(original_postings) != len(loaded_postings):
        print(f"Length mismatch: {len(original_postings)} vs {len(loaded_postings)}")
        return False

    for i, (expected, actual) in enumerate(zip(original_postings, loaded_postings)):
        if expected != actual:
            print(f"Posting list mismatch at position {i}")
            print(f"Expected: {expected}")
            print(f"Actual:   {actual}")
            return False

    print("Original and loaded indices match.")
    return True

In [28]:
# compression / loading / saving test
original_index = Index()
original_index.build_index(msmarco, tokenizer, max_num_docs=100, d_gap=True)

msmarco_index2 = Index()
msmarco_index2.build_index(msmarco, tokenizer, max_num_docs=100, d_gap=True)
msmarco_index2.compress_index("gamma")
msmarco_index2.save_index("msmarco_index_gamma.bin")


msmarco_index2.clear_index()
msmarco_index2.load_index("msmarco_index_gamma.bin")
print(check_correctness(original_index, msmarco_index2, with_tf=True))
msmarco_index2.compress_index("delta")
msmarco_index2.save_index("msmarco_index_delta.bin")


msmarco_index2.clear_index()
msmarco_index2.load_index("msmarco_index_varbyte.bin")
print(check_correctness(original_index, msmarco_index2, with_tf=True))

Loading index with compression code: 1, removed_tf_flag: 0


TypeError: check_correctness() got an unexpected keyword argument 'with_tf'

## Analysis between order and unordered docIds

In [ ]:

# print("First 20 :", reordered_inverted_index.index.items())


def create_SPIMI_index_ordered(tokenizer, k = 10):
    # build the index part
    msmarco_index = Index()
    msmarco_index.build_index(msmarco, tokenizer, d_gap = True, max_num_docs=10001)

    #build the matrix part
    mat, doc_ids, terms = create_sparse_tf_idf_matrix(msmarco_index)
    df = pd.DataFrame.sparse.from_spmatrix(mat, index=doc_ids, columns=terms)
    
    norm_df = normalize(df, norm="l2") #! cosa fa , è fondamentele se non la metti clusteing fa schifo 
    # find clusters and medoids
    kmeans = KMeans(n_clusters=k, random_state=0) #! use jaccard distance instead of euclidean
    clusters = kmeans.fit(norm_df)
    document_clusters = kmeans.labels_

    medoids = []
    # for each cluster, find the medoid document (closest to the cluster center)
    for i in range(kmeans.n_clusters):
        cluster_docs = np.flatnonzero(document_clusters == i)
        cluster_vectors = norm_df[cluster_docs]
        medoid_idx = np.argmin(np.linalg.norm(cluster_vectors - kmeans.cluster_centers_[i], axis=1))
        medoids.append(cluster_docs[medoid_idx])
    
    # calculate distance matrix between medoids
    medoid_vectors = norm_df[medoids]
    nonzero_cols = medoid_vectors.getnnz(axis=0) > 0
    medoid_vectors = medoid_vectors[:, nonzero_cols].toarray()
    dist_matrix = calculate_distance_matrix(norm_df, medoid_vectors, method="jaccard")

    tsp = nx.approximation.traveling_salesman_problem

    tsp_graph = nx.from_numpy_array(dist_matrix)
    tsp_path = tsp(tsp_graph, cycle=False)
    print("TSP path (medoid indices):", tsp_path)
    
    ordered_doc_ids, cluster_docs = reorder_documents_by_tsp(document_clusters, medoids, tsp_path)

    docs = list(islice(msmarco.docs_iter(), len(document_clusters)))

    class OrderedDataset:
        def __init__(self, docs, order):
            self.docs = docs
            self.order = order
        def docs_iter(self):
            for i in self.order:
                yield self.docs[i]

    ordered_dataset = OrderedDataset(docs, ordered_doc_ids)

    reordered_inverted_index = Index()
    reordered_inverted_index.build_index(
        ordered_dataset,
        tokenizer,
        max_num_docs=-1, #! -1 since reordered docs are already limited to 10001 in the original index
        d_gap=True
    )
    return reordered_inverted_index

    



In [20]:
import matplotlib.pyplot as plt
avg_gaps = []
k_values = [10, 20, 50, 100]

for k_ in k_values:
    reordered_inverted_index = create_SPIMI_index_ordered(tokenizer, k=k_)
    gaps = all_gaps(reordered_inverted_index)
    avg_gaps.append(np.mean(gaps))

plt.plot(k_values, avg_gaps, marker='o')
plt.xlabel("k")
plt.ylabel("Average d-gap")
plt.title("Average d-gap vs number of clusters")
plt.show()


c:\Users\Utente\Desktop\Magistrale\Information retrieval and web search\project_ir_\Info_Retrieval_proj\index\inverted_index.py:173: RuntimeWarning: divide by zero encountered in log10
  return np.log10(self.num_docs/df)


ValueError: Input contains infinity or a value too large for dtype('float64').

In [21]:
print(len(gaps_order))
print(len(gaps_unorderd))

380412
380412


In [ ]:


plt.hist(gaps_order, bins=50, alpha=0.5, label='Index ordered')
plt.hist(gaps_unorderd, bins=50, alpha=0.5, label='Index unordered')

plt.xlabel("Gap")
plt.ylabel("Frequency")
plt.legend()

plt.show()
import numpy as np

def plot_cdf(data, label):
    sorted_data = np.sort(data)
    y = np.arange(len(sorted_data)) / float(len(sorted_data))

    plt.plot(sorted_data, y, label=label)

plot_cdf(gaps_order, "ordered")
plot_cdf(gaps_unorderd, "unordered")

plt.xlabel("Gap")
plt.ylabel("CDF")
plt.legend()


plt.show()

KeyboardInterrupt: 

: 

: 